# TerraTree — Step 2: Feature Extraction & Training Table

This notebook:
1. Reconnects to Earth Engine and rebuilds the same AOI + composites from notebook 01
2. Computes spectral/red-edge indices (Sentinel-2) and SAR-derived features (Sentinel-1)
3. Builds a labeled reference map: Mangrove (from GMW) / Water / Other vegetation / Bare-mudflat (from ESA WorldCover)
4. Samples pixels + their feature values from that labeled map — this is your training table, replacing field ground-truth
5. Exports the table as a CSV to Google Drive

This recomputes the composites from scratch (fast — Earth Engine works with
image *definitions*, not downloaded pixels, until you explicitly export or
sample) rather than reading the GeoTIFFs from notebook 01. Cleaner and avoids
re-uploading large rasters into Colab.

In [ ]:
!pip install geemap -q

In [ ]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='terratree')

## Rebuild the AOI

Tries several name variants against OpenStreetMap in order, since the
first attempt in notebook 01 needed adjustment before it matched.

In [ ]:
import requests

QUERY_CANDIDATES = [
    "Sundarbans National Park, West Bengal, India",
    "Sundarban Tiger Reserve",
    "Sundarbans Biosphere Reserve",
    "Sundarbans",
]

aoi = None
for query in QUERY_CANDIDATES:
    resp = requests.get(
        "https://nominatim.openstreetmap.org/search",
        params={"q": query, "format": "geojson", "polygon_geojson": 1, "limit": 1},
        headers={"User-Agent": "terratree-major-project"}
    )
    features = resp.json().get('features', [])
    if features:
        aoi = ee.Geometry(features[0]['geometry'])
        print(f"Matched on query: '{query}'")
        break

if aoi is None:
    print("All OSM queries failed — using fallback bounding box.")
    aoi = ee.Geometry.Rectangle([88.75, 21.50, 89.20, 22.20])

## Rebuild the Sentinel-2 and Sentinel-1 composites

Same logic as notebook 01 — kept identical so features are computed on
the exact same imagery that was exported.

In [ ]:
START_DATE = '2022-01-01'
END_DATE = '2024-12-31'
MAX_CLOUD_PCT = 30

def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
           qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    return image.updateMask(mask).divide(10000).copyProperties(image, ['system:time_start'])

s2_collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAX_CLOUD_PCT))
    .map(mask_s2_clouds)
)
s2_median = s2_collection.median().clip(aoi)

s1_collection = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
)
s1_median = s1_collection.select(['VV', 'VH']).median().toFloat().clip(aoi)

print("Composites rebuilt.")

## Sentinel-2 feature extraction: spectral + red-edge indices

- **NDVI** — general vegetation greenness
- **NDWI** — water content, separates open water from land
- **NDVIre, NDI45, NDre1** — red-edge indices, sensitive to subtle canopy/species differences (following the reference paper's approach)

In [ ]:
def add_s2_indices(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = img.normalizedDifference(['B3', 'B8']).rename('NDWI')
    ndvire = img.normalizedDifference(['B8', 'B5']).rename('NDVIre')
    ndi45 = img.normalizedDifference(['B5', 'B4']).rename('NDI45')
    ndre1 = img.normalizedDifference(['B6', 'B5']).rename('NDre1')
    return img.addBands([ndvi, ndwi, ndvire, ndi45, ndre1])

s2_features = add_s2_indices(s2_median)
print("S2 feature bands:", s2_features.bandNames().getInfo())

## Sentinel-1 feature extraction: backscatter ratio + texture

- **VH/VV ratio** — a simple derived structural feature, cheap and often informative
- **GLCM texture (contrast, entropy)** on VV — captures canopy structural heterogeneity, following the "Texture Features" step in the original methodology

In [ ]:
vh_vv_ratio = s1_median.select('VH').divide(s1_median.select('VV')).rename('VH_VV_ratio')

# GLCM texture needs an integer input — scale VV to a reasonable 0-255-ish range first
vv_scaled = s1_median.select('VV').add(30).multiply(8).toInt32().rename('VV_scaled')
glcm = vv_scaled.glcmTexture(size=3)
texture_bands = glcm.select(['VV_scaled_contrast', 'VV_scaled_ent']).rename(['VV_contrast', 'VV_entropy'])

s1_features = s1_median.addBands([vh_vv_ratio, texture_bands])
print("S1 feature bands:", s1_features.bandNames().getInfo())

## Combine into one multi-source feature stack

This is the actual "fusion" step — one image where every pixel carries
both optical and radar-derived information together.

In [ ]:
feature_stack = s2_features.addBands(s1_features)
print("Combined feature stack bands:", feature_stack.bandNames().getInfo())

## Build the labeled reference map (replaces field ground-truth)

Priority order:
1. Start from ESA WorldCover reclassified into 3 broad groups (water, vegetation, bare/built)
2. Overwrite with GMW's mangrove mask wherever it says mangrove — GMW is the
   authoritative, purpose-built mangrove dataset, so it takes precedence over
   WorldCover's generic "tree cover" class in those pixels

Class codes: 0 = Water, 1 = Mangrove, 2 = Other vegetation, 3 = Bare/built/mudflat

In [ ]:
worldcover = ee.ImageCollection('ESA/WorldCover/v200').first().clip(aoi)

# WorldCover codes: 10 tree cover, 20 shrub, 30 grass, 40 crop, 50 built,
# 60 bare, 80 water, 90/95 wetland/mangrove (some versions), 100 moss
water_mask = worldcover.eq(80)
veg_mask = worldcover.eq(10).Or(worldcover.eq(20)).Or(worldcover.eq(30)).Or(worldcover.eq(40)).Or(worldcover.eq(90))
bare_mask = worldcover.eq(50).Or(worldcover.eq(60))

label = ee.Image(3).rename('label')  # default: bare/built
label = label.where(veg_mask, 2)
label = label.where(water_mask, 0)

try:
    gmw = ee.ImageCollection("projects/sat-io/open-datasets/GMW/extent/GMW_V3")
    gmw_latest = gmw.sort('system:time_start', False).first().clip(aoi)
except Exception:
    gmw_latest = ee.ImageCollection('LANDSAT/MANGROVE_FORESTS').mosaic().clip(aoi)

label = label.where(gmw_latest.gt(0), 1)  # mangrove overrides everything else
label = label.toInt().rename('label')

print("Label map built. Classes: 0=Water, 1=Mangrove, 2=Other vegetation, 3=Bare/built")

## Visual sanity check

Before sampling, look at this — if the mangrove class doesn't roughly
match the green coastal fringe you saw in notebook 01, something's off
and it's worth catching now rather than after training a model on bad labels.

In [ ]:
Map = geemap.Map(basemap='HYBRID')
Map.centerObject(aoi, 10)
Map.addLayer(label, {'min': 0, 'max': 3, 'palette': ['0000FF', '00FF00', '808000', 'A0522D']},
             'Labels (blue=water, green=mangrove, olive=other veg, brown=bare)')
Map

## Stratified sampling: build the training table

Samples a fixed number of pixels per class so the training set isn't
dominated by whichever class happens to cover the most area (water and
bare land usually dwarf mangrove in raw pixel count).

In [ ]:
SAMPLES_PER_CLASS = 1500

sample_image = feature_stack.addBands(label)

training_points = sample_image.stratifiedSample(
    numPoints=SAMPLES_PER_CLASS,
    classBand='label',
    region=aoi,
    scale=10,
    seed=42,
    geometries=True
)

print("Total sampled points:", training_points.size().getInfo())

## Export the training table to Google Drive

This CSV is what notebook 03 trains the Random Forest on.

In [ ]:
export_table = ee.batch.Export.table.toDrive(
    collection=training_points,
    description='sundarbans_training_table',
    folder='terratree',
    fileNamePrefix='sundarbans_training_table',
    fileFormat='CSV'
)
export_table.start()

print("Export started. Poll below.")

In [ ]:
import time

while True:
    state = export_table.status()['state']
    print(state)
    if state in {'COMPLETED', 'FAILED', 'CANCELLED'}:
        break
    time.sleep(20)

## Next steps
- [ ] Confirm the label map visual actually looks right (mangrove fringe roughly matches the coastline)
- [ ] Confirm `sundarbans_training_table.csv` landed in Drive
- [ ] Move to `03_model_training.ipynb` — load this CSV, do correlation-based feature selection, train Random Forest, evaluate with OA/Kappa/confusion matrix, add SHAP